In [ ]:
# load dataset
import pandas as pd

df = pd.read_csv("dataset.csv")

In [2]:
df.head()

,headline,label
0,cock suck before you piss around on my work,-1
1,you are gay or antisemmitian archangel white ...,-1
2,fuck your filthy mother in the ass dry,-1
3,get fuck ed up get fuck ed up got a drink t...,-1
4,stupid peace of shit stop deleting my stuff ...,-1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18148 entries, 0 to 18147
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   headline  18148 non-null  object
 1   label     18148 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 283.7+ KB


In [ ]:
df.isna().sum()  # drop NA if any

headline    0
label       0
dtype: int64

In [5]:
# preprocessing
# Select duplicate rows
duplicated_df = df[df.duplicated()]
len(duplicated_df)

1079

In [6]:
duplicated_df

,headline,label
302,every other press agency said the crowd was ch...,-1
1512,no abusing bots and stop being an idiot the pe...,-1
1681,and why is screw you mother fuck er a perso...,-1
2181,you better fuck off ruddy shit fuck,-1
2396,fuck off fuck off,-1
...,...,...
18094,TCS deke already mood ka bhosda hoga,-1
18095,Bhosda kar diya lavde subhe subhe,-1
18103,Mallana Ansar Raza chutiya or zahil hai....,-1
18121,Wo sala khud hii bheja hoga... taki 2 emotion ...,-1


In [7]:
# before dedup
len(df)

18148

In [8]:
df = df.drop_duplicates()

In [9]:
# after dedup
len(df)

17069

In [ ]:
df["label"].unique()

array([-1,  0])

In [ ]:
# data manipulation
# Labels: 1 = bullying, 0 = non-bullying
def perform_data_manipulation(df):
    df = df.copy()
    df["label"] = df["label"].replace(-1, 1)
    return df

In [12]:
df = perform_data_manipulation(df)

In [ ]:
df["label"].unique()

array([1, 0])

In [ ]:
df["label"].value_counts()

label
1    10588
0     6481
Name: count, dtype: int64

In [ ]:
# preprocessing

import re
import string
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
import contractions

# Build stopword set — REMOVE negation words so they are preserved
stop = set(stopwords.words("english"))
negation_words = {
    "not",
    "no",
    "never",
    "neither",
    "nor",
    "nothing",
    "nowhere",
    "nobody",
    "none",
    "cannot",
    "without",
    "against",
    "hardly",
    "scarcely",
    "barely",
    "doesnt",
    "isnt",
    "wasnt",
    "shouldnt",
    "wouldnt",
    "couldnt",
    "wont",
    "cant",
    "dont",
    "didnt",
    "hadnt",
    "hasnt",
    "havent",
    "neednt",
    "mightnt",
    "mustnt",
}
stop = stop - negation_words


def expand_contractions(text):
    """Expand contractions so negations are preserved: don't → do not."""
    return contractions.fix(text)


def negate_sequence(text):
    """
    Attach NOT_ prefix to every word that follows a negation term,
    until a clause-ending punctuation or 'but' is reached.
    E.g. 'do not hate you' → 'do not NOT_hate NOT_you'
    """
    negation_tokens = {
        "not",
        "no",
        "never",
        "nobody",
        "nothing",
        "nowhere",
        "neither",
        "nor",
        "cannot",
        "without",
        "hardly",
        "scarcely",
        "barely",
    }
    clause_breakers = {"but", "however", "although", "though", "yet"}

    tokens = text.split()
    result = []
    negating = False

    for token in tokens:
        # Strip trailing punctuation for matching, keep original for output
        clean_token = token.rstrip(".,!?;:")

        if clean_token in negation_tokens:
            negating = True
            result.append(token)
        elif clean_token in clause_breakers or token.endswith((".", "!", "?", ";")):
            negating = False
            result.append(token)
        elif negating:
            result.append("NOT_" + token)
        else:
            result.append(token)

    return " ".join(result)


def preprocess_text(text):
    wl = WordNetLemmatizer()

    # Remove HTML tags
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text()

    # Expand contractions FIRST so "don't" → "do not" before any removal
    text = expand_contractions(text)

    # Remove emojis
    emoji_clean = re.compile(
        "["
        "\U0001f600-\U0001f64f"
        "\U0001f300-\U0001f5ff"
        "\U0001f680-\U0001f6ff"
        "\U0001f1e0-\U0001f1ff"
        "\U00002702-\U000027b0"
        "\U000024c2-\U0001f251"
        "]+",
        flags=re.UNICODE,
    )
    text = emoji_clean.sub(r"", text)

    # Add space after full stop, remove URLs
    text = re.sub(r"\.(?=\S)", ". ", text)
    text = re.sub(r"http\S+", "", text)

    # Lowercase and remove punctuation (keep only letters and spaces)
    text = re.sub(r"[^a-zA-Z\s]", "", text.lower())

    # Apply negation tagging BEFORE stopword removal
    text = negate_sequence(text)

    # Remove stopwords and lemmatize
    # Keep NOT_ prefixed tokens even if the root is a stopword
    tokens = []
    for word in text.split():
        if word.startswith("NOT_"):
            root = word[4:]  # strip NOT_ for lemmatization
            if root.isalpha():
                tokens.append("NOT_" + wl.lemmatize(root))
        elif word not in stop and word.isalpha():
            tokens.append(wl.lemmatize(word))

    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
df["headline"][0]

'cock  suck before you piss around on my work'

In [ ]:
df["content"] = df["headline"].apply(preprocess_text)

In [ ]:
df["content"][0]

'cock suck piss around work'

In [20]:
# quick test to check negation
sanity_tests = [
    "Nobody likes you",
    "I do not hate you",
    "You are never good enough",
    "I never want to hurt you",
    "Don't be so mean",
    "I don't think you're stupid",
]
print("\n── Negation Sanity Check ──")
for t in sanity_tests:
    print(f"  Original : {t}")
    print(f"  Processed: {preprocess_text(t)}\n")


── Negation Sanity Check ──
  Original : Nobody likes you
  Processed: nobody NOT_like NOT_you

  Original : I do not hate you
  Processed: not NOT_hate NOT_you

  Original : You are never good enough
  Processed: never NOT_good NOT_enough

  Original : I never want to hurt you
  Processed: never NOT_want NOT_to NOT_hurt NOT_you

  Original : Don't be so mean
  Processed: not NOT_be NOT_so NOT_mean

  Original : I don't think you're stupid
  Processed: not NOT_think NOT_you NOT_are NOT_stupid



In [ ]:
words_len = df["content"].str.split().map(lambda x: len(x))
df_temp = df.copy()
df_temp["words length"] = words_len
df_temp

,headline,label,content,words length
0,cock suck before you piss around on my work,1,cock suck piss around work,5
1,you are gay or antisemmitian archangel white ...,1,gay antisemmitian archangel white tiger meow g...,106
2,fuck your filthy mother in the ass dry,1,fuck filthy mother as dry,5
3,get fuck ed up get fuck ed up got a drink t...,1,get fuck ed get fuck ed got drink cannot NOT_p...,25
4,stupid peace of shit stop deleting my stuff ...,1,stupid peace shit stop deleting stuff as hole ...,14
...,...,...,...,...
18143,deepak chahal se baga chutiya maine nahi dekha...,1,deepak chahal se baga chutiya maine nahi dekha...,11
18144,carry ki maa ki chut,1,carry ki maa ki chut,5
18145,ram kapoor ko priya se pyaar hai kya ?,0,ram kapoor ko priya se pyaar hai kya,8
18146,kya ram kapoor ki behen ke chut mai mera lund ...,1,kya ram kapoor ki behen ke chut mai mera lund ...,14


In [ ]:
max(df_temp["words length"])

1250

In [ ]:
from collections import Counter

words = " ".join(df["content"]).split()
counter = Counter(words)

# remove stopwords
filtered = {word: count for word, count in counter.items() if word not in stop}

# most common
most_common = sorted(filtered.items(), key=lambda x: x[1], reverse=True)

# least common
least_common = sorted(filtered.items(), key=lambda x: x[1])

print("Most common:", most_common[:20])
print("Least common:", least_common[:20])

Most common: [('fuck', 17463), ('NOT_the', 11543), ('NOT_a', 8505), ('NOT_to', 7799), ('NOT_you', 7519), ('not', 6751), ('NOT_and', 6525), ('NOT_i', 6023), ('NOT_of', 5393), ('suck', 5198), ('shit', 4622), ('NOT_is', 4526), ('NOT_that', 4246), ('NOT_it', 4147), ('NOT_in', 3727), ('nigger', 3439), ('ing', 3057), ('NOT_this', 2485), ('no', 2475), ('wikipedia', 2473)]
Least common: [('antisemmitian', 1), ('archangel', 1), ('greetingsh', 1), ('NOT_hasty', 1), ('NOT_mediocre', 1), ('NOT_mcdickerson', 1), ('NOT_shatter', 1), ('NOT_overweight', 1), ('NOT_womb', 1), ('ejaculating', 1), ('noble', 1), ('oife', 1), ('NOT_conndoms', 1), ('NOT_unhistorical', 1), ('NOT_pedantry', 1), ('NOT_sus', 1), ('blrude', 1), ('awakening', 1), ('leviathan', 1), ('poopie', 1)]


In [24]:
# Text encoding
from sklearn.model_selection import train_test_split

In [ ]:
x_data = df["content"]
y_data = df["label"]

In [27]:
# train/test split
x_train, x_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=0.2, random_state=42
)

In [28]:
# tf-idf vectorization
# ngram_range=(1,2) captures bigrams like "NOT_hate", "never bully"
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),  # unigrams + bigrams for negation context
    min_df=2,  # ignore very rare tokens
)

In [29]:
tfidf_vectorizer.fit(x_train)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word or character n-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21 Since v0.21, if ``input`` is ``'filename'`` or ``'file'``, the data is first read from the file and then passed to the given callable analyzer.",'word'
,"stop_words stop_words: {'english'}, list, default=NoneIf a string, it is passed to _check_stop_list and the appropriate stoplist is returned. 'english' is currently the only supported stringvalue.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp selects tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1

In [30]:
x_train_encoded = tfidf_vectorizer.transform(x_train)
x_test_encoded = tfidf_vectorizer.transform(x_test)

In [31]:
x_train_encoded.shape

(13655, 5000)

In [32]:
# model training and evaluation
import time
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC(),
    "MultinomialNB": MultinomialNB(),
    "SGD": SGDClassifier(),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
}

results = []

for name, model in models.items():
    start = time.time()
    model.fit(x_train_encoded, y_train)
    train_time = time.time() - start

    start = time.time()
    y_pred = model.predict(x_test_encoded)
    pred_time = time.time() - start

    results.append(
        [
            name,
            f1_score(y_test, y_pred, average="weighted"),
            precision_score(y_test, y_pred, average="weighted"),
            recall_score(y_test, y_pred, average="weighted"),
            train_time,
            pred_time,
        ]
    )

results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "F1 Score",
        "Precision",
        "Recall",
        "Train Time",
        "Prediction Time",
    ],
)
print(results_df.sort_values("F1 Score", ascending=False).to_string(index=False))

              Model  F1 Score  Precision   Recall  Train Time  Prediction Time
            XGBoost  0.917713   0.920523 0.917106   10.860645         0.031003
                SGD  0.915625   0.916261 0.915349    0.051095         0.000998
Logistic Regression  0.910585   0.910988 0.910369    0.168368         0.000000
          LinearSVC  0.905449   0.906270 0.905097    0.128223         0.000988
      MultinomialNB  0.869678   0.869840 0.870533    0.013117         0.001124


Selected model: SGD
Reasons: Almost the same accuracy as XGBoost, 200× faster training, very fast prediction, works perfectly with TF-IDF sparse vectors


In [33]:
best_model = SGDClassifier()
best_model.fit(x_train_encoded, y_train)
y_pred_best = best_model.predict(x_test_encoded)

In [34]:
print("\n── Classification Report (SGD) ──")
print(classification_report(y_test, y_pred_best))


── Classification Report (SGD) ──
              precision    recall  f1-score   support

           0       0.87      0.91      0.89      1299
           1       0.94      0.92      0.93      2115

    accuracy                           0.91      3414
   macro avg       0.91      0.91      0.91      3414
weighted avg       0.92      0.91      0.91      3414



In [35]:
# save model and vextorizer
import joblib
 
joblib.dump(best_model, "model.pkl")
joblib.dump(tfidf_vectorizer, "vectorizer.pkl")
print("Model and vectorizer saved.")

Model and vectorizer saved.


In [36]:
# Load saved model and vectorizer
model = joblib.load("model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

In [ ]:
# test some sentences
text = "Nobody likes you"

processedText = preprocess_text(text)
X = vectorizer.transform([text])
prediction = model.predict(X)[0]

if prediction == 1:
    print("Bullying detected")
else:
    print("Non-bullying content")

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
import joblib

joblib.dump(model, "cyberbullying_model.pkl")

In [ ]:
joblib.dump(tfidf_vectorizer, "vectorizer.pkl")

In [ ]:
# Load saved model and vectorizer
model = joblib.load("cyberbullying_model.pkl")
vectorizer = joblib.load("vectorizer.pkl")

In [ ]:
text = "Nobody likes you"

X = vectorizer.transform([text])
prediction = model.predict(X)[0]

if prediction == 1:
    print("Bullying detected")
else:
    print("Non-bullying content")

Examples:
You are so stupid and useless
Have a great day everyone!
Nobody likes you
Thank you for your help
